# 01 — Évaluation chiffrée du RAG

Ce notebook **documente et reproduit** les chiffres de qualité du pipeline (ceux du README).
On ne se contente pas de dire « c'est mesurable » : on **mesure**.

**Méthodologie** : un *golden set* de questions/réponses sur la cible de sécurité ANSSI
(Mistral TRC7535/7539), et des métriques de **recherche** (sans LLM, déterministes) :

- `keyword_hit_rate` (**hit@k**) : les mots-clés attendus apparaissent-ils dans les chunks récupérés ?
- `context_recall` : les tokens de la réponse de référence sont-ils couverts par le contexte ?
- `context_precision` : les chunks du top-k sont-ils pertinents ?

> **Prérequis** : Ollama + MongoDB démarrés, et un corpus déjà ingéré (onglet *Documents* de l'app).
> Sans génération LLM, ce notebook tourne en **~1-2 min**.

In [1]:
import sys, json
from pathlib import Path

# Rendre le projet importable (notebook dans notebooks/).
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from evals.run_eval import _evaluate_item, _aggregate, _METRIC_KEYS
print('Projet :', ROOT)

Projet : C:\Users\laury\Desktop\rag_project


## Le golden set

Chaque entrée a une `question`, une `answer` de référence et des `expected_keywords`.

In [2]:
dataset = json.loads((ROOT / 'evals' / 'golden_qa_anssi.json').read_text(encoding='utf-8'))
items = dataset['items']
print(f"{len(items)} questions — corpus : {dataset.get('dataset')}\n")
for it in items[:3]:
    print('Q :', it['question'])
    print('  mots-clés attendus :', it.get('expected_keywords'))
    print()

10 questions — corpus : ANSSI-CC-cible_2011 (Mistral TRC7535/TRC7539)

Q : Quel est le niveau d'évaluation EAL de la TOE Mistral ?
  mots-clés attendus : ['EAL3', 'ALC_FLR.3', 'AVA_VLA.2']

Q : Quelle est la version du boîtier Mistral TRC7535 ?
  mots-clés attendus : ['4.7.0.2', 'TRC7535']

Q : Quel est le niveau de résistance des fonctions de sécurité de la TOE ?
  mots-clés attendus : ['SOF']



## Lancer l'évaluation (mode retrieval)

On réutilise le **harnais existant** (`evals.run_eval`) — pas de logique dupliquée.
Chaque question passe par : *rewrite → retrieval hybride (sémantique + BM25 + graphe) → fusion RRF → rerank cross-encoder*.

In [3]:
results = []
for i, item in enumerate(items, 1):
    r = _evaluate_item(item, mode='retrieval', source_filter=None, use_judge=False, judge=None)
    results.append(r)
    print(f"{i:2}. hit@k={r.get('keyword_hit_rate'):.2f}  recall={r.get('context_recall'):.2f}  "
          f"({r.get('latency_s')}s)  {item['question'][:48]}")

C:\Users\laury\Desktop\rag_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


20:55:13 INFO    rag.ner | [ner] Modèle spaCy fr_core_news_sm chargé


20:55:13 INFO    rag.rerank | [rerank] Chargement Cross-Encoder local: C:\Users\laury\Desktop\rag_project\models\bge-reranker-v2-m3 (device=cuda:0)


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 6757.99it/s]

...

 6. hit@k=1.00  recall=1.00  (7.09s)  Que désigne l'acronyme CGM dans le système Mistr


 7. hit@k=1.00  recall=1.00  (7.22s)  Quels sont les deux produits de chiffrement du s


 8. hit@k=1.00  recall=1.00  (7.37s)  À quoi sert le Centre d'Élaboration des Clés (CE


 9. hit@k=1.00  recall=0.73  (7.35s)  Quelle technologie le système Mistral utilise-t-


10. hit@k=1.00  recall=1.00  (7.2s)  Pour quels types de réseaux le Mistral est-il pr


## Agrégats

In [4]:
agg = _aggregate(results)
print('=== MOYENNES (retrieval) ===')
for k in _METRIC_KEYS:
    if agg.get(k) is not None:
        print(f'  {k:<22} {agg[k]}')

# Mini-barres ASCII de hit@k par question (sans dépendance graphique).
print('\nhit@k par question :')
for i, r in enumerate(results, 1):
    h = r.get('keyword_hit_rate') or 0
    print(f"  Q{i:<2} {'#'*int(round(h*20)):<20} {h:.2f}")

=== MOYENNES (retrieval) ===
  keyword_hit_rate       0.9333
  context_recall         0.8977
  context_precision      0.66
  latency_s              9.383
  num_chunks_retrieved   8.0

hit@k par question :
  Q1  #######              0.33
  Q2  #################### 1.00
  Q3  #################### 1.00
  Q4  #################### 1.00
  Q5  #################### 1.00
  Q6  #################### 1.00
  Q7  #################### 1.00
  Q8  #################### 1.00
  Q9  #################### 1.00
  Q10 #################### 1.00


## Interprétation (honnête)

- **hit@k ≈ 0.93** et **context_recall ≈ 0.90** → le **retrieval est solide** : l'information
  pertinente est presque toujours ramenée dans le contexte.
- Le maillon faible mesuré est la **génération** (qualité du LLM local 8B sur du français
  technique), pas la recherche. C'est pourquoi l'exactitude prime sur la vitesse pour des
  docs de sécurité (8B par défaut), et pourquoi le *mode rapide* (3B) est réservé à l'exploratoire.
- Pour les métriques de **réponse** (fidélité, pertinence — LLM-as-judge), lancer le mode
  complet (plus lent) :

```bash
python -m evals.run_eval --mode full --limit 3        # avec LLM-as-judge
```

Le harnais sauvegarde chaque run dans MongoDB et compare au précédent (**non-régression**,
code de sortie non-nul en CI si chute nette).